# Module 8 — The Wealth-Signal Demo with Bounded Agent

**ATLAS: Aligned Three-Layer Architecture for Semantics**  
FSI (Financial Services Industry) Semantic Layer Workshop on AWS (Amazon Web Services)

---

## What this module teaches

This is the capstone. Every prior module built a piece of the architecture. Module 8
stitches them together into the workshop's reference application: a wealth-signal
detection workflow that identifies a Consumer-side customer with in-bank wealth
indicators, scores the lead, presents it to a bounded agent for routing, and surfaces
the routed lead to a human Wealth advisor for approval.

By the end of this module you can:

- Demonstrate the end-to-end wealth-signal workflow to a CIO (Chief Information Officer)
- Explain how the bounded agent selects routes from an enumerated set (not by LLM reasoning)
- Show the full audit trail from signal detection through advisor approval in SPARQL
- Explain why every component in the workflow is classified as deterministic,
  probabilistic-explainable, or probabilistic-opaque

## Key Terms for This Module

| Term | What It Is |
|------|------------|
| **Bounded agent** | A Step Functions state machine (optionally orchestrated by Bedrock AgentCore) that selects from a finite, declaratively-defined set of routing actions. The LLM inside the runtime interprets context but does not enlarge the route set. |
| **AWS Step Functions** | A serverless workflow orchestration service. You define a state machine (states + transitions), and Step Functions executes it. In ATLAS, the state machine IS the bounded agent. |
| **Human-in-the-loop (HITL)** | A workflow pattern where an automated process pauses for human review before a consequential action. In ATLAS, the advisor reviews and approves/declines each routed lead. |
| **Task token** | A Step Functions mechanism for pausing a workflow until an external system (the reviewer UI) signals completion. The workflow waits; the human decides; the workflow resumes. |
| **XGBoost** | Extreme Gradient Boosting — a gradient-boosted tree algorithm used in the SageMaker scoring path. Produces deterministic-given-version outputs that pair with SHAP for explainability. |
| **SHAP (SHapley Additive exPlanations)** | A feature-attribution method that explains which input features drove a model's output for a specific record. Required for every Score in ATLAS. |
| **Amazon Bedrock AgentCore** | AWS service for building agents with explicit tool declarations. In ATLAS, demonstrates the same bounded-agent pattern as Step Functions with a different orchestration surface. |
| **EventBridge** | Amazon EventBridge — a serverless event bus. In ATLAS, fires when a wealth-eligibility event lands in the LGD and when a routing decision is approved. |
| **AppSync** | AWS AppSync — a managed GraphQL service. Backs the reviewer UI that lists pending wealth leads with their evidence chains. |
| **Alex Morgan** | The synthetic Wealth advisor persona who reviews leads in the demo. The only named individual in the workshop. |

## The end-to-end workflow

```
1. EventBridge rule fires (wealth-eligibility event in LGD)
       ↓
2. Step Functions state machine (bounded agent) orchestrates:
   a. Enrich event with household context from SLGD
   b. Call SageMaker XGBoost endpoint for score + SHAP
   c. Invoke Bedrock to draft a contact note (for human review)
   d. Select routing target from enumerated set
   e. Pause for human approval (task token)
       ↓
3. Advisor (Alex Morgan) reviews in Streamlit UI:
   - Sees SHAP-attributed score
   - Sees in-bank evidence chain
   - Sees Bedrock-drafted contact note
   - Approves or declines with comment
       ↓
4. On approval:
   - State machine writes routing decision to SLGD with PROV-O
   - Fires outbound EventBridge event for downstream CRM
   - Closes the workflow
```

## The single rule that governs agents in ATLAS

> **LLMs do not make routing decisions; agents execute routing decisions, where the
> decision logic is deterministic and the LLM's role inside the agent is interface,
> not reasoning.**

The LLM inside the agent runtime:
- Reads tool outputs (SPARQL results, XGBoost scores)
- Helps the agent choose which tool to call next from a declared set
- Drafts narratives for the human reviewer

The LLM does NOT:
- Select routes by free reasoning
- Invent tools
- Make compliance decisions
- Enlarge the route set beyond what the state machine declares

## Prerequisites

- All prior modules complete (Modules 1–7)
- Neptune clusters running with ontology and promoted data
- Bedrock access for narrative drafting

## Deliverables

- A simulated end-to-end workflow execution
- The full audit trail queryable in SPARQL
- A demonstration script suitable for presenting to a CIO

## Architecture class for this module

**MIXED.** The workflow contains all three component classes:
- DETERMINISTIC: SHACL validation, route selection from enumerated set, SPARQL queries
- PROBABILISTIC-EXPLAINABLE: XGBoost scoring with SHAP attributions
- PROBABILISTIC-OPAQUE: Bedrock narrative drafting (interface role only, not a decision input)

In [ ]:
import sys
sys.path.insert(0, '../notebooks/shared')

import json
from datetime import datetime
from pathlib import Path
import random
import atlas_synthetic
import atlas_sparql

print('Module 8 — The Wealth-Signal Demo with Bounded Agent')
print(f'Synthetic data seed: {atlas_synthetic.ATLAS_SEED}')
print(f'Demo persona: Alex Morgan (Wealth Advisor)')

ATLAS_NS = 'https://github.com/your-org/atlas/ontology#'
INST_NS = 'https://github.com/your-org/atlas/instance#'
XSD_NS = 'http://www.w3.org/2001/XMLSchema#'
RDF_TYPE = 'http://www.w3.org/1999/02/22-rdf-syntax-ns#type'

## Simulating the End-to-End Workflow

In production, this workflow runs as a Step Functions state machine triggered by
EventBridge. For the workshop, we simulate each step in sequence to show what
happens at each stage and what data flows between components.

### Step 1: A wealth-eligibility event arrives

A customer's deposit-balance trajectory crosses the large-deposit threshold.
The transaction monitor fires an event to the LGD.

In [ ]:
# Step 1: A wealth-eligibility event arrives
rng = random.Random(atlas_synthetic.ATLAS_SEED)

# Pick a customer with a large-deposit signal
customers = atlas_synthetic.generate_customers(n=200)
accounts = atlas_synthetic.generate_accounts(customers)
transactions = atlas_synthetic.generate_transactions(accounts, lookback_days=90)

# Find a signal transaction
signal_txn = next(t for t in transactions if t.get('signal_tag') == 'large-deposit-pattern')
target_customer = next(c for c in customers if c['customer_id'] == signal_txn['customer_id'])

print('Step 1: Wealth-Eligibility Event Detected')
print('=' * 60)
print(f'  Customer:    {target_customer["first_name"]} {target_customer["last_name"]}')
print(f'  Customer ID: {target_customer["customer_id"][:12]}...')
print(f'  Segment:     {target_customer["segment"]}')
print(f'  Household:   {target_customer["household_id"][:12]}...')
print(f'  Signal type: large-deposit-pattern')
print(f'  Amount:      ${signal_txn["amount_usd"]:,.2f}')
print(f'  Date:        {signal_txn["transaction_date"]}')
print()
print('  Event written to LGD as atlas:BehavioralEvent')
print('  EventBridge rule fires -> Step Functions state machine starts')

### Step 2: The bounded agent enriches and scores

The state machine:
1. Queries the SLGD for household context (other members, combined balance)
2. Calls the SageMaker XGBoost endpoint for a wealth-conversion probability score
3. Attaches SHAP feature attributions to the score

In [ ]:
# Step 2: Enrich and Score
print('Step 2: Bounded Agent Enriches and Scores')
print('=' * 60)

# Simulate household enrichment
household_members = [c for c in customers if c['household_id'] == target_customer['household_id']]
household_balance = sum(
    a['balance_usd'] for a in accounts
    if a['customer_id'] in [m['customer_id'] for m in household_members]
)

print(f'  Household enrichment:')
print(f'    Members in household: {len(household_members)}')
print(f'    Combined balance:     ${household_balance:,.2f}')
print()

# Simulate XGBoost scoring
score_value = round(rng.uniform(0.72, 0.95), 3)
shap_features = {
    'deposit_amount': round(rng.uniform(0.15, 0.35), 3),
    'household_balance': round(rng.uniform(0.10, 0.25), 3),
    'account_tenure_years': round(rng.uniform(0.05, 0.15), 3),
    'segment_affluent': round(rng.uniform(0.08, 0.20), 3),
    'prior_surfacing_none': round(rng.uniform(0.02, 0.10), 3),
}

print(f'  XGBoost Score:')
print(f'    Wealth-conversion probability: {score_value}')
print(f'    Model version: wealth-xgb-v1.0')
print(f'    Component class: PROBABILISTIC-EXPLAINABLE')
print()
print(f'  SHAP Feature Attributions:')
for feature, contribution in sorted(shap_features.items(), key=lambda x: -x[1]):
    bar = '#' * int(contribution * 50)
    print(f'    {feature:<25} {contribution:.3f} {bar}')
print()
print(f'  Total SHAP sum: {sum(shap_features.values()):.3f}')

### Step 3: Route selection from the enumerated set

The bounded agent selects a route. The route set is **closed** — defined in the
state machine, enforced by the SHACL routing-policy shape from Module 6.

The three permissible routes:
- `ROUTE_ADVISOR_QUEUE` — send to advisor for review
- `ROUTE_SUPPRESSION_LIST` — do not contact (customer opted out or recently contacted)
- `ROUTE_ESCALATION` — escalate to senior advisor or compliance

The selection logic is deterministic: if score >= 0.7 and no prior suppression,
route to advisor queue. The LLM does not participate in this decision.

In [ ]:
# Step 3: Route Selection (DETERMINISTIC)
print('Step 3: Route Selection')
print('=' * 60)

# Deterministic routing logic (NOT LLM-driven)
if score_value >= 0.7:
    selected_route = 'ROUTE_ADVISOR_QUEUE'
    route_reason = f'Score {score_value} >= 0.7 threshold, no prior suppression'
elif score_value >= 0.5:
    selected_route = 'ROUTE_ESCALATION'
    route_reason = f'Score {score_value} in [0.5, 0.7) range, requires senior review'
else:
    selected_route = 'ROUTE_SUPPRESSION_LIST'
    route_reason = f'Score {score_value} < 0.5, below threshold'

print(f'  Selected route: {selected_route}')
print(f'  Reason:         {route_reason}')
print(f'  Component class: DETERMINISTIC (rule-based, not LLM)')
print()
print(f'  SHACL routing-policy shape check:')
print(f'    Route "{selected_route}" is in closed set: PASS')
print(f'    (Module 6 shape would reject any value not in the set)')

### Step 4: Human-in-the-loop review

The state machine pauses (via a task token) and presents the lead to the advisor.
Alex Morgan sees:
- The customer's name and segment
- The wealth-conversion score with SHAP attributions
- The in-bank evidence (the large deposit transaction)
- A Bedrock-drafted contact note (for review, not for direct use)

Alex approves or declines with a comment.

In [ ]:
# Step 4: Human Review (Alex Morgan)
print('Step 4: Human-in-the-Loop Review')
print('=' * 60)
print()
print('  Reviewer: Alex Morgan (Wealth Advisor)')
print('  Lead presented in reviewer UI:')
print(f'    Customer:  {target_customer["first_name"]} {target_customer["last_name"]}')
print(f'    Segment:   {target_customer["segment"]}')
print(f'    Score:     {score_value} (SHAP-explained)')
print(f'    Signal:    Large Deposit Pattern (${signal_txn["amount_usd"]:,.2f})')
print(f'    Route:     {selected_route}')
print()

# Simulate Bedrock drafting a contact note
contact_note = (
    f'{target_customer["first_name"]} {target_customer["last_name"]} recently deposited '
    f'${signal_txn["amount_usd"]:,.2f} into their checking account. Combined with a '
    f'household balance of ${household_balance:,.2f}, this suggests potential interest '
    f'in wealth management services. Recommend scheduling an introductory call.'
)
print(f'  Bedrock-drafted contact note (for review, not direct use):')
print(f'    "{contact_note}"')
print()
print(f'  Component class of contact note: PROBABILISTIC-OPAQUE')
print(f'  (Drafted by LLM, reviewed by human, not a compliance input)')
print()

# Simulate approval
review_outcome = 'APPROVED'
review_comment = 'Good candidate. Schedule intro call next week.'
review_timestamp = datetime.utcnow().isoformat() + 'Z'

print(f'  Alex Morgan\'s decision: {review_outcome}')
print(f'  Comment: "{review_comment}"')
print(f'  Timestamp: {review_timestamp}')
print()
print(f'  Task token released -> state machine resumes')

### Step 5: Write the audit trail to the SLGD

On approval, the state machine writes the complete routing decision to the SLGD
with full PROV-O provenance. This is the audit trail a regulator can query.

In [ ]:
# Step 5: Write Audit Trail to SLGD
print('Step 5: Audit Trail Written to SLGD')
print('=' * 60)

# Generate the audit trail triples
audit_triples = []
workflow_id = f'workflow-{datetime.now().strftime("%Y%m%d-%H%M%S")}'
cust_uri = f'<{INST_NS}customer-{target_customer["customer_id"]}>'
signal_uri = f'<{INST_NS}signal-{workflow_id}>'
score_uri = f'<{INST_NS}score-{workflow_id}>'
route_uri = f'<{INST_NS}routing-{workflow_id}>'
review_uri = f'<{INST_NS}review-{workflow_id}>'
advisor_uri = f'<{INST_NS}advisor-alex-morgan>'

# Signal
audit_triples.append(f'{signal_uri} <{RDF_TYPE}> <{ATLAS_NS}WealthSignal> .')
audit_triples.append(f'{signal_uri} <{ATLAS_NS}hasSignalType> <{ATLAS_NS}LargeDepositPattern> .')
audit_triples.append(f'{cust_uri} <{ATLAS_NS}producesSignal> {signal_uri} .')

# Score (probabilistic-explainable)
audit_triples.append(f'{score_uri} <{RDF_TYPE}> <{ATLAS_NS}Score> .')
audit_triples.append(f'{score_uri} <{ATLAS_NS}scoreValue> "{score_value}"^^<{XSD_NS}decimal> .')
audit_triples.append(f'{score_uri} <{ATLAS_NS}probabilistic> "true"^^<{XSD_NS}boolean> .')
audit_triples.append(f'{score_uri} <{ATLAS_NS}explainability> "true"^^<{XSD_NS}boolean> .')
audit_triples.append(f'{score_uri} <{ATLAS_NS}modelVersion> "wealth-xgb-v1.0"^^<{XSD_NS}string> .')
audit_triples.append(f'{score_uri} <{ATLAS_NS}confidence> "{score_value}"^^<{XSD_NS}decimal> .')
audit_triples.append(f'{signal_uri} <{ATLAS_NS}hasScore> {score_uri} .')

# Routing decision (deterministic)
audit_triples.append(f'{route_uri} <{RDF_TYPE}> <{ATLAS_NS}RoutingDecision> .')
audit_triples.append(f'{route_uri} <{ATLAS_NS}selectedRoute> "{selected_route}"^^<{XSD_NS}string> .')

# Human review
audit_triples.append(f'{review_uri} <{RDF_TYPE}> <{ATLAS_NS}HumanReview> .')
audit_triples.append(f'{review_uri} <{ATLAS_NS}reviewOutcome> "{review_outcome}"^^<{XSD_NS}string> .')
audit_triples.append(f'{review_uri} <{ATLAS_NS}reviewDate> "{review_timestamp}"^^<{XSD_NS}dateTime> .')
audit_triples.append(f'{route_uri} <{ATLAS_NS}reviewedBy> {review_uri} .')
audit_triples.append(f'{review_uri} <{ATLAS_NS}conductedBy> {advisor_uri} .')

# Advisor
audit_triples.append(f'{advisor_uri} <{RDF_TYPE}> <{ATLAS_NS}Advisor> .')
audit_triples.append(f'{advisor_uri} <http://www.w3.org/2000/01/rdf-schema#label> "Alex Morgan"^^<{XSD_NS}string> .')

print(f'  Audit trail triples generated: {len(audit_triples)}')
print()
print(f'  The complete chain:')
print(f'    Customer -> producesSignal -> WealthSignal')
print(f'    WealthSignal -> hasScore -> Score (probabilistic-explainable)')
print(f'    RoutingDecision -> selectedRoute -> ROUTE_ADVISOR_QUEUE')
print(f'    RoutingDecision -> reviewedBy -> HumanReview')
print(f'    HumanReview -> conductedBy -> Advisor (Alex Morgan)')
print(f'    HumanReview -> reviewOutcome -> APPROVED')
print()
print(f'  This is queryable via SPARQL (the CQ6 audit-trail query from Module 1).')

## The CIO Demo: One Query That Shows Everything

This is what you show a CIO at the end of the workshop. One SPARQL query that
traverses the entire audit trail from signal detection through advisor approval:

In [ ]:
# The CIO Demo Query
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, XSD as XSD_NS_RDF

ATLAS_RDF = Namespace('https://github.com/your-org/atlas/ontology#')
INST_RDF = Namespace('https://github.com/your-org/atlas/instance#')

# Build the audit trail as a local graph (simulating SLGD query)
g_demo = Graph()
for triple_str in audit_triples:
    # Parse N-Triple into the graph
    parts = triple_str.rstrip(' .').split(' ', 2)
    if len(parts) == 3:
        s = URIRef(parts[0].strip('<>'))
        p = URIRef(parts[1].strip('<>'))
        if parts[2].startswith('<'):
            o = URIRef(parts[2].strip('<>'))
        elif '^^' in parts[2]:
            val, dtype = parts[2].rsplit('^^', 1)
            val = val.strip('"')
            dtype = URIRef(dtype.strip('<>'))
            o = Literal(val, datatype=dtype)
        else:
            o = Literal(parts[2].strip('"'))
        g_demo.add((s, p, o))

# Run the CIO demo query
cio_query = atlas_sparql.build_prefixes() + '''
SELECT ?customer ?signal ?signalType ?score ?scoreValue ?route ?reviewOutcome ?advisor WHERE {
    ?customer atlas:producesSignal ?signal .
    ?signal atlas:hasSignalType ?signalType ;
            atlas:hasScore ?score .
    ?score atlas:scoreValue ?scoreValue .
    ?routing atlas:selectedRoute ?route ;
             atlas:reviewedBy ?review .
    ?review atlas:reviewOutcome ?reviewOutcome ;
            atlas:conductedBy ?advisor .
}
'''

print('The CIO Demo Query')
print('=' * 60)
print()
print('"Show me the complete path from signal detection to advisor approval."')
print()

results = list(g_demo.query(atlas_sparql.validate(cio_query)))
if results:
    for row in results:
        print(f'  Signal type:    {str(row.signalType).split("#")[-1]}')
        print(f'  Score:          {row.scoreValue}')
        print(f'  Route:          {row.route}')
        print(f'  Review outcome: {row.reviewOutcome}')
        print(f'  Advisor:        {str(row.advisor).split("#")[-1]}')
else:
    print('  (No results - check audit trail construction)')

print()
print('This is what the reader demos to a CIO at the end of Module 8.')
print('One query. Full audit trail. Every component classified.')

In [ ]:
print('=' * 60)
print('MODULE 8 VALIDATION GATE')
print('=' * 60)
print()

gate_pass = True

# Gate 1: Audit trail has all required components
required_types = ['WealthSignal', 'Score', 'RoutingDecision', 'HumanReview', 'Advisor']
for rtype in required_types:
    found = any(rtype in t for t in audit_triples)
    status = 'PASS' if found else 'FAIL'
    print(f'[{status}] Gate 1.{required_types.index(rtype)+1} - {rtype} in audit trail')
    if not found: gate_pass = False

# Gate 2: Score has SHAP (explainability = true)
has_explainability = any('explainability' in t and 'true' in t for t in audit_triples)
print(f'[{"PASS" if has_explainability else "FAIL"}] Gate 2 - Score has explainability=true (SHAP)')
if not has_explainability: gate_pass = False

# Gate 3: Route is from closed set
has_valid_route = any('ROUTE_ADVISOR_QUEUE' in t or 'ROUTE_SUPPRESSION_LIST' in t or 'ROUTE_ESCALATION' in t for t in audit_triples)
print(f'[{"PASS" if has_valid_route else "FAIL"}] Gate 3 - Route from closed enumerated set')
if not has_valid_route: gate_pass = False

# Gate 4: Human review has outcome and advisor
has_outcome = any('reviewOutcome' in t for t in audit_triples)
has_advisor = any('conductedBy' in t for t in audit_triples)
print(f'[{"PASS" if has_outcome and has_advisor else "FAIL"}] Gate 4 - HumanReview has outcome + advisor')
if not (has_outcome and has_advisor): gate_pass = False

# Gate 5: CIO query returns results
print(f'[{"PASS" if results else "FAIL"}] Gate 5 - CIO demo query returns results')
if not results: gate_pass = False

print()
if gate_pass:
    print('MODULE 8 VALIDATION: PASS')
    print()
    print('Congratulations. You have completed the ATLAS workshop.')
    print('You can now:')
    print('  1. Articulate why the deterministic-vs-probabilistic boundary matters')
    print('  2. Build this pattern in your own account against your own data')
    print('  3. Defend the architecture in front of an MRM reviewer')
    print('  4. Extend the ontology with your institution\'s concepts')
else:
    print('MODULE 8 VALIDATION: FAIL')
    raise AssertionError('Module 8 validation gate failed.')

## What Changed

| Artifact | Location | Description |
|----------|----------|-------------|
| End-to-end workflow | This notebook | Simulated wealth-signal detection through advisor approval |
| Audit trail | Generated in cell 12 | Full PROV-O-attributed chain queryable via SPARQL |
| CIO demo query | Cell 14 | One query showing the complete path |

**The architecture is complete.**

Every layer is operational:
- **Data Integration**: Three patterns feeding the LGD (Module 4)
- **Ontology and Digital Twin**: FIBO-aligned, SHACL-validated, two-tier Neptune (Modules 1-3, 5-6)
- **Application**: Bounded agent, human-in-the-loop, NL-to-SPARQL (Modules 7-8)

The deterministic-vs-probabilistic boundary is enforced mechanically at every layer.
A reviewer can run the SHACL validator and produce a report. The audit trail is
queryable end-to-end. The architecture is defensible.

## Next Steps

- **Run the cleanup notebook** (`99-cleanup`) to tear down all infrastructure
- **Read `docs/follow-on-labs.md`** for the real-time depth lab and external-signals lab
- **Replace the synthetic data** with your institution's data using the Extending
  This to Your Data appendices from each module
- **Present to your CIO** using the demo script from this module